# 11. Supplementary tables and figures

This notebook draws **Supplementary Figs. 1–3 and Supplementary Tables 1–7** of the Supplementary Information from the aggregate results of notebooks 02–09 (it reads only `RESULTS_DIR`, never row-level data). Each item has its own section, with its title and legend as in the Supplementary Information in the markdown cell above the code. Tables are saved as CSV under `OUTPUT_DIR/tables`, figures as PNG (400 dpi) and PDF under `OUTPUT_DIR/figures`.

As in the Supplementary Information, a table is laid out with one header row per cohort (or per cohort and diagnosis), followed by one row per model.

In [ ]:
%run ./00_settings.ipynb
from matplotlib.patches import Rectangle

R = load_all()                                             # aggregate results (notebook 00, section 0.12)
WHO = (("black-white", "Black"), ("hispanic-white", "Hispanic"), ("asian-white", "Asian"))

## Supplementary Fig. 1 | Derivation of the two cohorts

EchoNext was used as released, and the MIMIC-IV cohort was assembled for this study. Group counts refer to patients. Patients recorded as other or unknown were used for fitting but not compared. ECG, electrocardiogram; LVEF, left ventricular ejection fraction; TTE, transthoracic echocardiogram.

In [ ]:
def box(ax, x, y, w, h, lines, fill="#f4f7fb", edge=AXIS, sizes=(8.2, 7.2), n_bold=1):
    """Rounded box with centred lines; the first n_bold lines are bold."""
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.004,rounding_size=0.012", fc=fill, ec=edge, lw=0.8, transform=ax.transAxes, clip_on=False))
    n = len(lines)
    for i, t in enumerate(lines):
        yy = y + h * (1 - (i + 0.5) / n) if n > 1 else y + h / 2
        ax.text(x + w / 2, yy, t, transform=ax.transAxes, ha="center", va="center", fontsize=sizes[0] if i < n_bold else sizes[1],
                color=INK if i < n_bold else INK2, fontweight="bold" if i < n_bold else "normal")


def arrow(ax, x0, y0, x1, y1):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0), xycoords="axes fraction", textcoords="axes fraction",
                arrowprops=dict(arrowstyle="-|>", color=MUTED, lw=1.0, shrinkA=0, shrinkB=0))


def figure_s1(R):
    ch = R["ch"].set_index(["cohort", "group"])
    flow = load_json(RESULTS_DIR / "03_mimic" / "flow.json")["flow"]           # steps of notebook 03, section 3.2
    final = load_json(RESULTS_DIR / "03_mimic" / "final_cohort.json")          # after the waveform check (section 3.4)
    groups = ["Hispanic", "White", "Black", "Asian"]
    BW = 0.46                                                                   # width of a column of boxes
    fig, ax = plt.subplots(figsize=(7.2, 7.2)); ax.axis("off")

    def column(x, title, items, colour):
        """One column of boxes joined by arrows; the last box (the final cohort) has the cohort colour."""
        ax.text(x + BW / 2, 0.985, title, fontsize=9.2, fontweight="bold", color=INK, ha="center", transform=ax.transAxes)
        y = 0.94
        for k, (t, s, excl) in enumerate(items):
            box(ax, x, y - 0.078, BW, 0.078, [t, s], fill="#ffffff", edge=colour if k == len(items) - 1 else AXIS, sizes=(7.2, 6.8))
            if k < len(items) - 1:
                arrow(ax, x + BW / 2, y - 0.081, x + BW / 2, y - 0.119)
                if excl:
                    ax.text(x + BW / 2 + 0.012, y - 0.1, excl, fontsize=6.2, color=MUTED, va="center", transform=ax.transAxes)
            y -= 0.121
        return y

    e = ch.loc[("EchoNext", "All")]
    e_items = [("EchoNext-Mini release (PhysioNet v1.1.1)", f"{int(e.ecgs):,} ECGs · {int(e.patients):,} patients", "used as released,\nno further exclusions"),
               ("Final EchoNext cohort (5-fold analyses)", f"{int(e.ecgs):,} ECGs · {int(e.patients):,} patients", "")]
    ye = column(0.0, "EchoNext (primary cohort)", e_items, CE)
    ax.text(BW / 2, ye + 0.022, " · ".join(f"{gname} {int(ch.loc[('EchoNext', gname.lower())].patients):,}" for gname in groups), fontsize=6.8, color=INK2,
            ha="center", transform=ax.transAxes)
    m_items = [(f["step"], f"{f['ecgs']:,} ECGs · {f['patients']:,} patients", "") for f in flow]
    m_items[-1] = m_items[-1][:2] + ("waveform check: missing\nsamples or flat lead",)       # exclusion between the last step and the final cohort
    m_items.append(("Final MIMIC-IV cohort (5-fold analyses)", f"{final['n_ecgs']:,} ECGs · {final['n_patients']:,} patients", ""))
    ym = column(0.54, "MIMIC-IV (replication cohort)", m_items, CM)
    ax.text(0.54 + BW / 2, ym + 0.022, " · ".join(f"{gname} {final['group_patients'][gname.lower()]:,}" for gname in groups), fontsize=6.8, color=INK2,
            ha="center", transform=ax.transAxes)
    save_figure(fig, "SupplementaryFig1")


figure_s1(R)

## Supplementary Fig. 2 | Patient-level five-fold cross-fitting

Five folds were stratified by race and ethnicity and by composite structural heart disease. In each round one fold was evaluated, and the other patients were split into fitting and validation sets, with all electrocardiograms (ECGs) of a patient in the same set. The design was the same in both cohorts and for every analysis.

In [ ]:
def figure_s2():
    """Schematic of the design of notebooks 02 and 03 (grey scale: the cohort colours are not used for roles)."""
    fig, ax = plt.subplots(figsize=(7.2, 2.9)); ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    FIT, VAL, EVAL = "#e1e0d9", "#a9a79f", "#52514e"
    x0, w, h, gap = 0.105, 0.5, 0.108, 0.006
    ww = w / 5 - gap
    ax.text(x0, 0.985, "All patients of a cohort, divided into five folds", fontsize=8.2, fontweight="bold", color=INK, va="center")
    for f in range(5):
        ax.text(x0 + f * w / 5 + ww / 2, 0.895, f"Fold {'ABCDE'[f]}", ha="center", va="center", fontsize=7.6, color=INK2)
    for k in range(5):
        yy = 0.72 - k * 0.148
        ax.text(x0 - 0.012, yy + h / 2, f"Round {k + 1}", ha="right", va="center", fontsize=7.8, color=INK)
        for f in range(5):
            xx = x0 + f * w / 5
            if f == k:                                                          # the evaluated fold of this round
                ax.add_patch(Rectangle((xx, yy), ww, h, fc=EVAL, ec="none"))
                ax.text(xx + ww / 2, yy + h / 2, "evaluation", ha="center", va="center", fontsize=7.0, color="white", fontweight="bold")
            else:                                                               # the other folds: fitting (85%) and validation (15%)
                ax.add_patch(Rectangle((xx, yy), ww * 0.85, h, fc=FIT, ec="none"))
                ax.add_patch(Rectangle((xx + ww * 0.85, yy), ww * 0.15, h, fc=VAL, ec="none"))
    lx = 0.645
    for i, (c, t, s) in enumerate(((FIT, "Fitting (68% of patients)", "all ECGs of each patient"),
                                   (VAL, "Validation (12%)", "most recent ECG: regularisation strength, threshold"),
                                   (EVAL, "Evaluation (20%)", "most recent ECG, scored once"))):
        yy = 0.79 - i * 0.17
        ax.add_patch(Rectangle((lx, yy - 0.035), 0.028, 0.07, fc=c, ec="none"))
        ax.text(lx + 0.04, yy + 0.028, t, va="center", fontsize=7.8, color=INK, fontweight="bold")
        ax.text(lx + 0.04, yy - 0.032, s, va="center", fontsize=7.0, color=INK2)
    ax.text(lx, 0.2, "Every patient is evaluated exactly once, by probes that\nnever saw any ECG of that patient. The predictions of\nthe five rounds are pooled.",
            va="center", fontsize=7.4, color=INK, linespacing=1.35)
    save_figure(fig, "SupplementaryFig2")


figure_s2()

## Supplementary Fig. 3 | Detectors of composite SHD refitted after equalising prevalence across groups in the fitting data

Difference from White patients in flagging rate at the same age and sex, with 95% confidence intervals, for detectors fitted on all fitting ECGs (filled markers), on fitting ECGs subsampled so that every group had the prevalence of its age and sex stratum (open markers, value at the right) and on a random subsample of the same size that kept the original prevalences, which shows the effect of the smaller fitting set alone (grey diamonds). Positive values mean more frequent flagging than in White patients. ECG, electrocardiogram; SHD, structural heart disease.

In [ ]:
def figure_s3(R):
    X = R["x2"]
    X = X[X.fm.isin(ORDER) & (X.measure == "FLAG")]
    blocks = (("EchoNext", "black-white", "EchoNext, Black − White"), ("MIMIC-IV", "black-white", "MIMIC-IV, Black − White"),
              ("MIMIC-IV", "asian-white", "MIMIC-IV, Asian − White"))
    fig = plt.figure(figsize=(6.6, 4.3))
    ax = fig.add_axes([0.205, 0.215, 0.775, 0.765]); style(ax)
    ax.axvline(0, color=INK2, lw=0.8)
    y, ticks, labs, out = 0.0, [], [], {}
    for coh, pair, name in blocks:
        ax.text(-13.7, y - 0.86, name, fontsize=7.8, fontweight="bold", color=INK, va="center", ha="left")
        for fm in ORDER:
            for var, dy in (("all rows", -0.24), ("equalised", 0.0), ("random subsample", 0.24)):
                r = X[(X.cohort == coh) & (X.pair == pair) & (X.fm == fm) & (X.variant == var)].iloc[0]
                col = MUTED if var == "random subsample" else COH[coh]
                ax.plot([100 * r.lo, 100 * r.hi], [y + dy] * 2, color=col, lw=1.3 if var != "random subsample" else 0.9, solid_capstyle="round")
                ax.plot(100 * r["diff"], y + dy, "D" if var == "random subsample" else ("o" if coh == "EchoNext" else "s"), ms=3.2 if var == "random subsample" else 4.8,
                        mfc="white" if var == "equalised" else col, mec=col, mew=1.1)
                if var == "equalised":                                          # the values quoted in the text
                    ax.text(10.4, y + dy, f"{100 * r['diff']:+.1f}".replace("-", MINUS), fontsize=6.8, color=INK, va="center", ha="right")
                    out[f"{coh} | {pair} | {fm}"] = round(100 * float(r["diff"]), 1)
            ticks.append(y); labs.append(fm); y += 1
        y += 0.95
    ax.set_yticks(ticks); ax.set_yticklabels(labs, fontsize=7.6, color=INK)
    ax.set_ylim(y - 1.35, -1.4); ax.set_xlim(-14, 10.6); ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.text(10.4, -1.05, "equalised", fontsize=6.6, color=INK2, va="center", ha="right")
    ax.set_xlabel("Difference from White patients in flagging rate\n(percentage points, same age and sex)", fontsize=7.8, color=INK2)
    h = [Line2D([], [], marker="o", color=INK2, mfc=INK2, lw=1.3, ms=4.8, label="Fitted on all ECGs"),
         Line2D([], [], marker="o", color=INK2, mfc="white", lw=1.3, ms=4.8, label="Prevalence equalised in the fitting data (value shown)"),
         Line2D([], [], marker="D", color=MUTED, mfc=MUTED, lw=0.9, ms=3.2, label="Random subsample of the same size")]
    fig.legend(handles=h, loc="lower center", bbox_to_anchor=(0.55, 0.0), ncol=1, frameon=False, fontsize=7.0, handletextpad=0.5, labelspacing=0.25)
    save_figure(fig, "SupplementaryFig3")
    return out


pd.Series(figure_s3(R), name="flagging-rate difference after equalising (percentage points)").to_frame()

## Supplementary Table 1 | Foundation models

ECG, electrocardiogram.

All models were used with their published weights and without fine-tuning, and the largest of the three HuBERT-ECG sizes was used. Parameter counts exclude modules needed only for pretraining. No model used race labels or EchoNext data in pretraining.

The parameter counts and embedding dimensions are read from notebook 04 (section 4.4), which counts the parameters of every network as it is used for the embeddings; the other columns describe the pretraining as reported by the authors of each model.

In [ ]:
models = load_json(RESULTS_DIR / "04_embeddings" / "models.json")
mp = lambda k: f"{models[k]['parameters'] / 1e6:.1f} million"
dim = lambda k: f"{models[k]['dim']:,}"
tableS1 = save_table(pd.DataFrame([
    ["ECGFounder", f"Convolutional network, Net1D ({mp('ecgfounder')})", "Supervised, 150 diagnostic labels",
     "10.8 million ECGs from the Harvard-Emory ECG Database. MIMIC-IV-ECG not included", dim("ecgfounder")],
    ["ECG-FM", f"Transformer, wav2vec 2.0 type ({mp('ecgfm')})", "Self-supervised, contrastive and masked objectives",
     "PhysioNet 2021 challenge datasets and MIMIC-IV-ECG", dim("ecgfm")],
    ["HuBERT-ECG", f"Transformer, HuBERT, large version ({mp('hubert_large')})", "Self-supervised, masked prediction of cluster labels",
     "9.1 million ECGs from 11 datasets, including MIMIC-IV-ECG", dim("hubert_large")]],
    columns=["Model", "Architecture (parameters)", "Pretraining objective", "Pretraining data", "Embedding dimension"]), "SupplementaryTable1")
tableS1

## Supplementary Table 2 | Echocardiographic labels in MIMIC-IV: proportion of studies in which the label could be determined, and use in the analysis

ECG, electrocardiogram; LV, left ventricular; LVEF, left ventricular ejection fraction; RV, right ventricular; TR, tricuspid regurgitation.

Computed on the echocardiogram linked to the most recent ECG of each patient. All studies also include patients recorded as other or unknown. LVEF was the reported value, or the biplane or three-dimensional estimate if no visual estimate was recorded, and wall thickness was the larger of the septal and inferolateral thickness. Mitral and tricuspid regurgitation were positive for grades of moderate (2+) or greater, and right ventricular dysfunction for moderate or severe hypokinesis. A label was used if it could be determined in at least 85% of studies in every group and in both reporting systems, with a difference between groups below 5 percentage points.

In [ ]:
S2_NAMES = {"lvef_le45": "LVEF ≤45%", "lvwt_ge13": "LV wall thickness ≥1.3 cm", "as_modplus": "Aortic stenosis (grade or measurements)",
            "mr_modplus": "Mitral regurgitation", "tr_modplus": "Tricuspid regurgitation", "rv_modplus": "RV systolic dysfunction",
            "ph_modplus": "Pulmonary hypertension (reported severity)", "trv_ge32": "TR velocity ≥3.2 m/s", "ar_modplus": "Aortic regurgitation",
            "pr_modplus": "Pulmonary regurgitation", "pe_modplus": "Pericardial effusion"}
asc = read_result("03_mimic", "ascertainment.csv")
main_components = load_json(RESULTS_DIR / "03_mimic" / "labels.json")["main_components"]
rows = []
for r in asc.itertuples():
    low_g, low_s = min(r.white, r.black, r.hispanic, r.asian), min(r.system_A, r.system_B)
    assert (r.label_key in main_components) == (low_g >= 85 and low_s >= 85 and r.max_diff_race < 5), r.label_key   # the rule stated in the notes
    rows.append([S2_NAMES[r.label_key], f"{r.overall:.1f}", f"{low_g:.1f}", f"{r.max_diff_race:.1f}", f"{low_s:.1f}",
                 "Main analysis" if r.label_key in main_components else "Not used"])
tableS2 = save_table(pd.DataFrame(rows, columns=["Label", "All studies, %", "Lowest of the four groups, %", "Largest difference between groups, points",
                                                 "Lower of the two reporting systems, %", "Use"]), "SupplementaryTable2")
tableS2

## Supplementary Table 3 | Decodability of race and ethnicity from embeddings

AUROC, area under the receiver operating characteristic curve; CI, confidence interval; ECG, electrocardiogram.

Random weights refers to the same architecture without pretraining, and After LEACE to a probe refitted after least-squares concept erasure (LEACE) of race and ethnicity information. The last four columns are one-versus-rest AUROCs of the four-group probe.

In [ ]:
ra = R["race_after"].set_index(["cohort", "fm"])
rows = []
for c, s1 in (("EchoNext", R["s1e"]), ("MIMIC-IV", R["s1m"])):
    rows.append([c] + [""] * 8)                                                # header row of the cohort
    for fm in ORDER:
        r, rr = s1.loc[KEY[fm]], s1.loc[KEY[fm] + "_random"]
        rows.append([fm, ci3(r.bw_auroc, r.bw_lo, r.bw_hi), ci3(rr.bw_auroc, rr.bw_lo, rr.bw_hi), f3(ra.loc[(c, fm), "race_bw_auroc"]),
                     ci3(r.macro_auroc, r.macro_lo, r.macro_hi), f3(r.ovr_white), f3(r.ovr_black), f3(r.ovr_hispanic), f3(r.ovr_asian)])
    r = s1.loc["meas_lr"]
    rows.append(["ECG measurements, age, sex", ci3(r.bw_auroc, r.bw_lo, r.bw_hi), "", "", ci3(r.macro_auroc, r.macro_lo, r.macro_hi), f3(r.ovr_white),
                 f3(r.ovr_black), f3(r.ovr_hispanic), f3(r.ovr_asian)])
tableS3 = save_table(pd.DataFrame(rows, columns=["Encoder", "Black vs White AUROC (95% CI)", "Random weights (95% CI)", "After LEACE",
                                                 "Four-group mean AUROC (95% CI)", "White", "Black", "Hispanic", "Asian"]), "SupplementaryTable3")
tableS3

## Supplementary Table 4 | Black vs White decodability after adjustment for cardiac structure

AUROC, area under the receiver operating characteristic curve; CI, confidence interval; SHD, structural heart disease.

Values are AUROCs except for matched pairs (their numbers are printed below the table), where the value is the proportion of pairs in which the Black patient had the higher probe score. Removal of structural components used patients with complete structural variables (numbers printed below the table), so the value before removal, like the first value for structural variables, differs slightly from that for all patients.

In [ ]:
rows, note = [], []
for c, S, s1 in (("EchoNext", R["e2"], R["s1e"]), ("MIMIC-IV", R["m2"], R["s1m"])):
    rows.append([c] + [""] * 4)
    for fm in ORDER:
        A_, B_ = S["A"].loc[KEY[fm]], S["B"].loc[KEY[fm]]
        cb, cr = (S["C"][(S["C"].model == KEY[fm]) & (S["C"].variant == v)].iloc[0] for v in ("baseline", "removed"))
        rows.append([fm, f3(s1.loc[KEY[fm], "bw_auroc"]), ci3(A_.bw_auroc, A_.bw_lo, A_.bw_hi), ci3(B_.pair_concordance, B_.lo, B_.hi),
                     mn(f"{cb.bw_auroc:.3f} to {cr.bw_auroc:.3f} ({cr.delta:+.3f}; {cr.delta_lo:+.3f}, {cr.delta_hi:+.3f})")])
    m0, bs = S["m0"].loc["structure only"], S["B"].loc["structure only"]
    rows.append(["Structural variables only", ci3(m0.bw_auroc, m0.bw_lo, m0.bw_hi), "", ci3(bs.pair_concordance, bs.lo, bs.hi), ""])
    mt, Cc = S["sum"]["matching"], S["C"].iloc[0]
    note.append(f"{c}: {mt['n_pairs']:,} matched pairs (of {mt['n_a_eligible']:,} eligible Black patients; largest |SMD| after matching "
                f"{mt['max_abs_smd_after']:.2f}); structural components removed in {int(Cc.n_black):,} Black and {int(Cc.n_white):,} White patients")
tableS4 = save_table(pd.DataFrame(rows, columns=["Model", "All patients", "No recorded SHD (95% CI)", "Matched pairs, concordance (95% CI)",
                                                 "Structural components removed: before to after (change; 95% CI)"]), "SupplementaryTable4")
print("\n".join(note))
tableS4

## Supplementary Table 5 | Detectors of composite SHD and aortic stenosis in Black and White patients: discrimination, and differences in sensitivity and false-positive rate at the same age and sex

AUROC, area under the receiver operating characteristic curve; CI, confidence interval; ECG, electrocardiogram; FPR, false-positive rate; SHD, structural heart disease.

Differences are Black minus White in percentage points, at the common threshold of each detector on the most recent ECG of each patient. AUROC by group is computed within the group, and differences at the same age and sex are directly standardised. The flagging-rate difference in Fig. 2b is the mean of the sensitivity and FPR differences, and the balanced-accuracy difference is half of the sensitivity difference minus the FPR difference.

In [ ]:
def diff_ci(c, lab, measure, flip=False):
    """Black - White difference with its 95% CI for every model; flip turns a difference in FNR into a difference in sensitivity."""
    d = tmq(R, c, lab, measure)
    assert len(d) == N_FM
    v, lo, hi = (-d.value, -d.hi, -d.lo) if flip else (d.value, d.lo, d.hi)
    return {fm: f"{signed(100 * v[fm])} ({signed(100 * lo[fm])}, {signed(100 * hi[fm])})" for fm in ORDER}


disc = R["disc"].set_index(["cohort", "label_key", "fm"])
rows = []
for c, comp, as_ in (("EchoNext", E_COMP, E_AS), ("MIMIC-IV", M_COMP, M_AS)):
    for lab, name in ((comp, "composite SHD"), (as_, "aortic stenosis")):
        rows.append([f"{c}, {name}"] + [""] * 5)
        sens, fpr = diff_ci(c, lab, "FNR", flip=True), diff_ci(c, lab, "FPR")
        for fm in ORDER:
            d = disc.loc[(c, lab, fm)]
            rows.append([fm, f3(d.auroc), f3(d.auroc_white), f3(d.auroc_black), sens[fm], fpr[fm]])
tableS5 = save_table(pd.DataFrame(rows, columns=["Model", "AUROC, all patients", "AUROC, White", "AUROC, Black",
                                                 "Sensitivity difference, same age and sex (95% CI)", "FPR difference, same age and sex (95% CI)"]), "SupplementaryTable5")
tableS5

## Supplementary Table 6 | Differences from White patients in flagging rate and in balanced accuracy for every diagnosis (means of the three models, percentage points, same age and sex)

FPR, false-positive rate; LV, left ventricular; LVEF, left ventricular ejection fraction; PASP, pulmonary artery systolic pressure; RV, right ventricular; SHD, structural heart disease; TR, tricuspid regurgitation.

Positive values mean a higher value than in White patients, and n is the number of patients with the diagnosis. Parentheses give the number of models, of three, whose 95% confidence interval excluded zero. Flagging rate is the mean of sensitivity and FPR, and balanced accuracy the mean of sensitivity and specificity, so the sensitivity difference equals the sum of the two values and the FPR difference equals the flagging-rate difference minus the balanced-accuracy difference. Comparisons with fewer than 10 patients in a denominator were not evaluated.

In [ ]:
T = R["tm"]
T = T[(T.eraser == "none")]


def mean_cell(c, lab, pair, measure):
    """Mean of the three models and the number of models whose 95% CI excluded zero; empty when the comparison was not evaluated."""
    d = T[(T.cohort == c) & (T.label_key == lab) & (T.pair == pair) & (T.measure == measure) & T.valid]
    if d.empty:
        return ""
    assert len(d) == N_FM, (c, lab, pair, measure, len(d))
    return f"{signed(100 * d.value.mean())} ({int(((d.lo > 0) | (d.hi < 0)).sum())}/{N_FM})"


rows = []
for c in ("EchoNext", "MIMIC-IV"):
    rows.append([c] + [""] * 9)
    for lab in T[T.cohort == c].label_key.drop_duplicates():
        r = [LABEL_NAME[lab]]
        for pair, _ in WHO:
            d = T[(T.cohort == c) & (T.label_key == lab) & (T.pair == pair)]
            r += [f"{int(d.pos_a.iloc[0]):,}", mean_cell(c, lab, pair, "FLAG"), mean_cell(c, lab, pair, "BA")]
        rows.append(r)
tableS6 = save_table(pd.DataFrame(rows, columns=["Diagnosis"] + [h for _, w in WHO for h in (f"{w}, n", f"{w} − White: flagging rate",
                                                                                           f"{w} − White: balanced accuracy")]), "SupplementaryTable6")
tableS6

## Supplementary Table 7 | Erasure of race information: discrimination of the detectors, and changes in the differences between Black and White patients in missed patients and false-positive results at the same age and sex (percentage points)

AUROC, area under the receiver operating characteristic curve; CI, confidence interval; ECG, electrocardiogram; FNR, false-negative rate; FPR, false-positive rate; LEACE, least-squares concept erasure; SHD, structural heart disease.

Differences are Black minus White, and changes are after minus before, with 95% confidence intervals from a paired bootstrap. Standard erasure is LEACE fitted on all fitting ECGs, and disease-negative erasure is LEACE fitted on fitting ECGs without the diagnosis of the row. All differences are standardised for age and sex, and a positive change in the FNR difference means that more Black patients were missed relative to White patients. Changes in flagging rate and balanced accuracy after standard erasure are shown in Fig. 3.

In [ ]:
def change_ci(c, lab, measure, eraser):
    """Change after erasure (after minus before) of the Black - White difference, with its paired 95% CI, for every model."""
    d = tmq(R, c, lab, measure, eraser=eraser)
    assert len(d) == N_FM
    return {fm: f"{signed(100 * d.loc[fm, 'change'])} ({signed(100 * d.loc[fm, 'change_lo'])}, {signed(100 * d.loc[fm, 'change_hi'])})" for fm in ORDER}


dauc = R["dauc"].set_index(["cohort", "label_key", "fm"])
rows = []
for c, comp, as_ in (("EchoNext", E_COMP, E_AS), ("MIMIC-IV", M_COMP, M_AS)):
    for lab, name, neg in ((comp, "composite SHD", "LEACE (SHD-neg fit)"), (as_, "aortic stenosis", "LEACE (AS-neg fit)")):
        rows.append([f"{c}, {name}"] + [""] * 6)
        fn, fp = change_ci(c, lab, "FNR", "LEACE"), change_ci(c, lab, "FPR", "LEACE")
        nfn, nfp, nba = (change_ci(c, lab, m, neg) for m in ("FNR", "FPR", "BA"))
        for fm in ORDER:
            a = dauc.loc[(c, lab, fm)]
            rows.append([fm, mn(f"{a.auroc_after - a.auroc_before:+.3f}"), fn[fm], fp[fm], nfn[fm], nfp[fm], nba[fm]])
tableS7 = save_table(pd.DataFrame(rows, columns=["Model", "Detector AUROC: change after standard erasure", "Standard erasure. FNR difference: change (95% CI)",
                                                 "Standard erasure. FPR difference: change (95% CI)", "Disease-negative erasure. FNR difference: change (95% CI)",
                                                 "Disease-negative erasure. FPR difference: change (95% CI)",
                                                 "Disease-negative erasure. Balanced-accuracy difference: change (95% CI)"]), "SupplementaryTable7")
tableS7